# Entrenament model offline - Predicció glucosa

#### Import de les llibreries necessaries

In [12]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


import warnings

warnings.simplefilter("ignore", FutureWarning)

pd.set_option('display.max_columns', None)

#### Definició de les columnes dels datasets

In [13]:
cols = [
    'year', 'month', 'day', 'hour', 'minute', 'second', # A–F
    'glucose_level', # G
    'finger_stick', # H
    'basal', # I
    'bolus', # J
    'sleep', # K
    'work', # L
    'stressors', # M
    'hypo_event', # N
    'illness', # O
    'exercise', # P
    'basis_heart_rate', # Q
    'basis_gsr', # R
    'basis_skin_temperature', # S
    'basis_air_temperature',  # T
    'basis_step', # U
    'basis_sleep', # V
    'meal', # W
    'meal_type' # X
]

#### Definició de pacients i horitzons

In [ ]:
# Definicio de numero dels pacients
PACIENTS = [559, 563, 570, 575, 588, 591]

# Definició de l'horitzo
HORITZO = {30:6, 60:12}  # minuts : files (pas de 5 mins)

## Carrega del dataset

In [15]:
# Funcio per obtenir els datasets dels pacients
def load_data(pacient, train_or_test):
    df=pd.read_csv(f'../data/{pacient}/{pacient}_{train_or_test}.csv', sep=';', header = None, names = cols)
    return df

prova_559_train = load_data(559, 'train')
prova_559_test = load_data(559, 'test')

## Preprocessament del dataset

In [ ]:
def preprocess(df):
    prep = df.copy()

    prep['time'] = pd.to_datetime(
        dict(year=df.year, month=df.month, day=df.day, hour=df.hour, minute=df.minute)
    )
    
    prep.sort_values('time', inplace=True)

    # Coma decimal a punt
    convert = ["basal","bolus","basis_gsr","basis_skin_temperature","basis_air_temperature"]
    
    for c in convert:
        prep[c] = (prep[c].astype(str)
                   .str.replace(",",".", regex=False)
                   .str.strip()
                   .astype(float))

    # Unifiquem el tipo de meal que han fet
    cat_meal = {
        1:"Desayuno",
        2:"Almuerzo",
        3:"Cena",
        4:"Snack",
        5:"Correccion_hipo"
    }

    prep["meal_type"] = prep["meal_type"].map(cat_meal).astype("category")
    prep = pd.get_dummies(prep, columns=['meal_type'], dummy_na=False, prefix='meal')

    # Drop columnas amb casi tot NaN o valor constant
    prep = prep.drop(columns=["second","finger_stick","meal"])

    # Zeros que no poden ser valids
    invalid_zero = [
        "glucose_level",
        "basis_heart_rate",
        "basis_gsr",
        "basis_skin_temperature",
        "basis_air_temperature"
    ]
    
    prep[invalid_zero] = prep[invalid_zero].replace(0, np.nan)

    # Fem servir forward-fill per tal d'asegurarnos que no es mira al futur
    prep[invalid_zero] = prep[invalid_zero].fillna(method='ffill') 

    prep = prep.dropna(subset=['glucose_level'])

    prep = prep.drop(columns=['time'])
    return prep

#### Proves de visualització del dataset

In [17]:
prova_559_train = preprocess(prova_559_train)
prova_559_test = preprocess(prova_559_test)

print('Shape train: ', prova_559_train.shape)
print('Shape test: ', prova_559_test.shape)
print('\n')
print(prova_559_train.isnull().mean()*100)

Shape train:  (12081, 25)
Shape test:  (2876, 24)


year                      0.000000
month                     0.000000
day                       0.000000
hour                      0.000000
minute                    0.000000
glucose_level             0.000000
basal                     0.000000
bolus                     0.000000
sleep                     0.000000
work                      0.000000
stressors                 0.000000
hypo_event                0.000000
illness                   0.000000
exercise                  0.000000
basis_heart_rate          1.158844
basis_gsr                 1.158844
basis_skin_temperature    1.158844
basis_air_temperature     1.158844
basis_step                0.000000
basis_sleep               0.000000
meal_Almuerzo             0.000000
meal_Cena                 0.000000
meal_Correccion_hipo      0.000000
meal_Desayuno             0.000000
meal_Snack                0.000000
dtype: float64


## Entrenament del model

#### Definició de la funcio de split features i target

In [18]:
def make_xy(df, files):

    # Definim y com al nivell de glucosa a predir
    # Agafarem el valor a pedir "x" files més amunt segons l'horitzo (30 mins: 6 files o 60 min: 12 files)
    y = df['glucose_level'].shift(-files)

    # Definim X amb totes les columnes pero sense les ultimes files, ja que no tindran predicció y
    X = df.iloc[:-files].copy()

    # Ajustem la y perque tingui el mateix nombre de files que x (eliminant les ultimes files ja que no poden ser predites)
    y = y.iloc[:-files]

    return X, y

#### Definició de la funció de evaluació del model

In [19]:
def evaluate(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    return rmse, mae

### Entrenament per cada pacient (model offline)

In [20]:
resultats = []

for pacient in PACIENTS:
    print(f'\nPacient {pacient}')
    train_raw = load_data(pacient, 'train')
    test_raw  = load_data(pacient, 'test')

    train = preprocess(train_raw)
    test  = preprocess(test_raw)

    # Igualem les columnes del train y del test i emplenem el que falti amb 0
    train, test = train.align(test, join='outer', axis=1, fill_value=0)

    # Ignorem els 60 minuts primers del test
    test = test.iloc[12:].reset_index(drop=True)

    for minuts, steps in HORITZO.items():
        # Entrenament del model
        X_train, y_train = make_xy(train, steps)

        model = RandomForestRegressor(
            n_estimators=1000,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        # Predicció del test
        X_test = test.iloc[:-steps].copy()
        y_true = test['glucose_level'].shift(-steps).iloc[:-steps].reset_index(drop=True)


        y_pred = model.predict(X_test)

        # guardem la predicció a csv
        directori_pred = f'../data/predicted/pred{pacient}_{minuts}min.csv'

        # Fem index més 12 per tal de poder comparar les prediccions visualment amb més facilitat
        df_pred = pd.DataFrame({'Index_del_test': X_test.index + 12, f'pred_glucosa_t+{minuts}': y_pred})
        df_pred.to_csv(directori_pred, index=False)

        # Evaluació del model
        rmse, mae = evaluate(y_true, y_pred)
        resultats.append({
            'Pacient': pacient,
            'Horitzo': minuts,
            'RMSE': rmse,
            'MAE' : mae
        })
        print(f'{minuts} min: RMSE={rmse:.2f}  MAE={mae:.2f}')




Pacient 559
30 min: RMSE=24.38  MAE=16.86
60 min: RMSE=38.11  MAE=27.94

Pacient 563
30 min: RMSE=20.84  MAE=15.29
60 min: RMSE=33.62  MAE=24.80

Pacient 570
30 min: RMSE=18.67  MAE=13.21
60 min: RMSE=31.21  MAE=23.26

Pacient 575
30 min: RMSE=24.48  MAE=18.14
60 min: RMSE=40.74  MAE=32.11

Pacient 588
30 min: RMSE=21.79  MAE=15.87
60 min: RMSE=33.41  MAE=24.59

Pacient 591
30 min: RMSE=25.63  MAE=19.15
60 min: RMSE=39.21  MAE=31.87


## Taula final de resutats

In [21]:
resultats_df = pd.DataFrame(resultats)

taula = (resultats_df.pivot(index='Pacient', columns='Horitzo', values=['RMSE','MAE']))
print(f'Visualització inicial de la taula: \n{taula}')

# Renombrem les columnes de la taula
taula.columns = ['RMSE 30','RMSE 60','MAE 30', 'MAE 60']

# Reordenem les columnes
taula = taula[['RMSE 30','MAE 30','RMSE 60','MAE 60']]

# Calculem el promig
promig = taula.mean().to_frame().T
promig.index = ['PROMIG'] 

# Mostrem la taula final amb el promig
taula_final = pd.concat([taula, promig], axis=0)

print("\nRESULTATS FINALS:")
print(taula_final)

Visualització inicial de la taula: 
              RMSE                   MAE           
Horitzo         30         60         30         60
Pacient                                            
559      24.378597  38.111626  16.863942  27.943685
563      20.838670  33.624143  15.285437  24.796404
570      18.674437  31.213704  13.207952  23.258335
575      24.481174  40.739030  18.144336  32.112006
588      21.794221  33.414295  15.865652  24.587457
591      25.633635  39.210914  19.151151  31.868775

RESULTATS FINALS:
          RMSE 30     MAE 30    RMSE 60     MAE 60
559     24.378597  16.863942  38.111626  27.943685
563     20.838670  15.285437  33.624143  24.796404
570     18.674437  13.207952  31.213704  23.258335
575     24.481174  18.144336  40.739030  32.112006
588     21.794221  15.865652  33.414295  24.587457
591     25.633635  19.151151  39.210914  31.868775
PROMIG  22.633456  16.419745  36.052285  27.427777
